In [1]:
%pip install pandas sidrapy
import pandas as pd
from sidrapy import get_table
import unidecode

def tratar_nomes_colunas(df):
    df.columns = [unidecode.unidecode(col) for col in df.columns]
    df.columns = [col.lower() for col in df.columns]
    df.columns = [col.replace(" ", "_") for col in df.columns]
    df.columns = [col.replace("-", "_") for col in df.columns]
    df.columns = [col.replace("(", "") for col in df.columns]
    df.columns = [col.replace(")", "") for col in df.columns]
    df.columns = [col.replace(".", "") for col in df.columns]
    df.columns = [col.replace(",", "") for col in df.columns]
    df.columns = [col.replace("'", "") for col in df.columns]
    df.columns = [col.replace("'", "") for col in df.columns]
    return df

def get_pop_data(codigo_municipio):
    # de 2000 a 2024 - sem 2007 e 2010
    data_geral = get_table(
        table_code="6579",
        territorial_level="6",
        ibge_territorial_code=codigo_municipio,
        period="all",
        variable="all",
        header="y",
    )
    data_geral.columns = data_geral.iloc[0]
    data_geral = data_geral.drop(data_geral.index[0])
    # incluindo a contagem de 2007, outra tabela no ibge
    data_2007 = get_table(
        table_code="793",
        territorial_level="6",
        ibge_territorial_code=codigo_municipio,
        period="all",
        variable="all",
        header="y",
    )
    data_2007.columns = data_2007.iloc[0]
    data_2007 = data_2007.drop(data_2007.index[0])
    # incluindo censo 2010, outra tabela no ibge
    data_2010 = get_table(
        table_code="608",
        territorial_level="6",
        ibge_territorial_code=codigo_municipio,
        period="all",
        variable="93",
        header="y",
    )
    data_2010.columns = data_2010.iloc[0]
    data_2010 = data_2010.drop(data_2010.index[0])
    data_2010 = data_2010.drop(
        columns=[
            "Situação do domicílio (Código)",
            "Situação do domicílio",
            "Sexo (Código)",
            "Sexo",
        ]
    )
    # concat
    data = pd.concat([data_geral, data_2007, data_2010], axis=0)
    data["Valor"] = data["Valor"].str.replace("-", "0").astype(int)
    data = tratar_nomes_colunas(data)
    return data


codigo_municipios = [
    "3534401", # Osaco
    "3552205", # Sorocaba
    "3543402", # Ribeirão Preto
    "3548708", # São Bernardo do Campo
    "3549904", # São José dos Campos
    "3547809"  # Santo André
]
dict_dfs = {}
for municipio in codigo_municipios:
    dict_dfs[municipio] = get_pop_data(municipio)

# consolidação e export para csv
pop = pd.concat(dict_dfs.values(), axis=0, ignore_index=True)

StatementMeta(, db01a900-5fe6-47cf-9555-e32e8c4d3c84, 8, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [2]:
# Conversão spark + save na bronze
spark_df = spark.createDataFrame(pop)
spark_df.write.mode("overwrite").format("delta").saveAsTable("gold_osasco_populacao_ibge")

StatementMeta(, db01a900-5fe6-47cf-9555-e32e8c4d3c84, 10, Finished, Available, Finished, False)